Fidel handwritten: data setup and CRNN baseline (train and test on Fidel).

In [ ]:
#@title Setup, dataset, vocabulary, loaders, and shared helpers

!pip install -q transformers datasets huggingface_hub evaluate jiwer pillow opencv-python sentencepiece accelerate

from google.colab import drive

import os
import random
import zipfile
import json
import time
import copy
import math

import numpy as np
import pandas as pd

from PIL import Image, ImageOps, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from huggingface_hub import hf_hub_download

from jiwer import cer, wer


drive.mount("/content/drive")


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


RESEARCH_ROOT = "/content/drive/MyDrive/amharic_htr_research"

OCR_ROOT = f"{RESEARCH_ROOT}/ocr"
OCR_RESEARCH_ROOT = f"{OCR_ROOT}/research"
SHARED_ROOT = f"{OCR_ROOT}/shared"
DATA_ROOT = f"{SHARED_ROOT}/data"
STATE_ROOT = f"{SHARED_ROOT}/state"

LOCAL_ROOT = "/content/amharic_htr"
LOCAL_DATA_ROOT = f"{LOCAL_ROOT}/data"

RESEARCH_DIRS = [
    f"{OCR_RESEARCH_ROOT}/crnn/extended",
    f"{OCR_RESEARCH_ROOT}/crnn/augmentation",
    f"{OCR_RESEARCH_ROOT}/crnn/beam_search",
    f"{OCR_RESEARCH_ROOT}/transformer/vit_htr",
    f"{OCR_RESEARCH_ROOT}/transformer/vit_augmentation",
    f"{OCR_RESEARCH_ROOT}/trocr/finetuning",
    f"{OCR_RESEARCH_ROOT}/hybrid/cnn_transformer",
    f"{OCR_RESEARCH_ROOT}/encoder_decoder/attention",
    f"{OCR_RESEARCH_ROOT}/cross_script",
    f"{OCR_RESEARCH_ROOT}/final_evaluation"
]

for directory in RESEARCH_DIRS:
    os.makedirs(directory, exist_ok=True)

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(STATE_ROOT, exist_ok=True)
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)


labels_path = hf_hub_download(
    repo_id="upanzi/fidel-dataset",
    filename="train_labels.csv",
    repo_type="dataset"
)

df = pd.read_csv(labels_path)

df["writer"] = df["writer"].astype(str)
df["image_filename"] = df["image_filename"].astype(str)
df["line_text"] = df["line_text"].fillna("").astype(str)

df = df[
    (df["type"] == "handwritten") &
    (df["line_text"].str.strip() != "")
].reset_index(drop=True)


zip_path = hf_hub_download(
    repo_id="upanzi/fidel-dataset",
    filename="train-clean.zip",
    repo_type="dataset"
)

extract_dir = f"{LOCAL_DATA_ROOT}/fidel_train"

os.makedirs(
    extract_dir,
    exist_ok=True
)

marker = f"{extract_dir}/.extracted"

if not os.path.exists(marker):

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        z.extractall(extract_dir)

    with open(marker, "w") as f:
        f.write("done")


image_map = {}

for root, dirs, files in os.walk(extract_dir):

    for filename in files:

        if filename.lower().endswith(
            (".png", ".jpg", ".jpeg")
        ):

            image_map[filename] = os.path.join(
                root,
                filename
            )


df["image_path"] = df[
    "image_filename"
].map(image_map)

df = df.dropna(
    subset=["image_path"]
).reset_index(drop=True)


train_split_path = f"{DATA_ROOT}/train_split.csv"
val_split_path = f"{DATA_ROOT}/validation_split.csv"

if (
    os.path.exists(train_split_path)
    and
    os.path.exists(val_split_path)
):

    train_df = pd.read_csv(
        train_split_path
    )

    val_df = pd.read_csv(
        val_split_path
    )

    for frame in [train_df, val_df]:

        frame["writer"] = frame[
            "writer"
        ].astype(str)

        frame["image_filename"] = frame[
            "image_filename"
        ].astype(str)

        frame["line_text"] = frame[
            "line_text"
        ].fillna("").astype(str)

        frame["image_path"] = frame[
            "image_filename"
        ].map(image_map)

else:

    writers = list(
        df["writer"].unique()
    )

    random.Random(SEED).shuffle(
        writers
    )

    val_writer_count = max(
        1,
        int(len(writers) * 0.10)
    )

    val_writers = set(
        writers[:val_writer_count]
    )

    train_writers = set(
        writers[val_writer_count:]
    )

    train_df = df[
        df["writer"].isin(train_writers)
    ].reset_index(drop=True)

    val_df = df[
        df["writer"].isin(val_writers)
    ].reset_index(drop=True)

    train_df.to_csv(
        train_split_path,
        index=False
    )

    val_df.to_csv(
        val_split_path,
        index=False
    )


train_writers = set(
    train_df["writer"].unique()
)

val_writers = set(
    val_df["writer"].unique()
)

assert train_writers.isdisjoint(
    val_writers
)


vocab_path = f"{DATA_ROOT}/vocabulary.json"

if os.path.exists(vocab_path):

    with open(
        vocab_path,
        "r",
        encoding="utf-8"
    ) as f:

        vocabulary = json.load(f)

    characters = vocabulary["characters"]
    blank_idx = vocabulary["blank_idx"]

    char_to_idx = {
        char: int(index)
        for char, index in vocabulary[
            "char_to_idx"
        ].items()
    }

    idx_to_char = {
        int(index): char
        for index, char in vocabulary[
            "idx_to_char"
        ].items()
    }

else:

    characters = sorted(
        set(
            "".join(
                train_df["line_text"]
            )
        )
    )

    blank_idx = 0

    char_to_idx = {
        char: i + 1
        for i, char in enumerate(characters)
    }

    idx_to_char = {
        i: char
        for char, i in char_to_idx.items()
    }

    vocabulary = {
        "characters": characters,
        "blank_idx": blank_idx,
        "char_to_idx": char_to_idx,
        "idx_to_char": {
            str(i): char
            for i, char in idx_to_char.items()
        }
    }

    with open(
        vocab_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            vocabulary,
            f,
            ensure_ascii=False,
            indent=2
        )


IMG_HEIGHT = 64
IMG_WIDTH = 512
BATCH_SIZE = 16


class FidelDataset(Dataset):

    def __init__(
        self,
        dataframe,
        augment=False
    ):

        self.df = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("L")

        if self.augment:

            if random.random() < 0.35:
                image = ImageEnhance.Contrast(
                    image
                ).enhance(
                    random.uniform(0.7, 1.3)
                )

            if random.random() < 0.25:
                image = ImageEnhance.Brightness(
                    image
                ).enhance(
                    random.uniform(0.8, 1.2)
                )

            if random.random() < 0.20:
                image = ImageOps.autocontrast(image)

        image = image.resize(
            (IMG_WIDTH, IMG_HEIGHT)
        )

        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.tensor(
            image
        ).unsqueeze(0)

        target = torch.tensor(
            [
                char_to_idx[c]
                for c in row["line_text"]
                if c in char_to_idx
            ],
            dtype=torch.long
        )

        return image, target


def collate_fn(batch):

    images = torch.stack(
        [x[0] for x in batch]
    )

    targets = torch.cat(
        [x[1] for x in batch]
    )

    target_lengths = torch.tensor(
        [len(x[1]) for x in batch],
        dtype=torch.long
    )

    return images, targets, target_lengths


train_dataset = FidelDataset(
    train_df,
    augment=False
)

val_dataset = FidelDataset(
    val_df,
    augment=False
)

aug_train_dataset = FidelDataset(
    train_df,
    augment=True
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

aug_train_loader = DataLoader(
    aug_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


def greedy_decode(outputs):

    predictions = outputs.argmax(dim=2)

    results = []

    for sequence in predictions.permute(1, 0):

        text = []
        previous = blank_idx

        for index in sequence.tolist():

            if (
                index != blank_idx
                and index != previous
            ):

                text.append(
                    idx_to_char.get(index, "")
                )

            previous = index

        results.append(
            "".join(text)
        )

    return results


def decode_targets(
    targets,
    target_lengths
):

    results = []
    offset = 0

    for length in target_lengths.tolist():

        target = targets[
            offset:offset + length
        ]

        results.append(
            "".join(
                idx_to_char.get(
                    value.item(),
                    ""
                )
                for value in target
            )
        )

        offset += length

    return results


def evaluate_model(
    model,
    loader,
    decoder=greedy_decode
):

    model.eval()

    predictions = []
    references = []

    start_time = time.time()

    with torch.no_grad():

        for images, targets, target_lengths in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(images)

            predictions.extend(
                decoder(outputs)
            )

            references.extend(
                decode_targets(
                    targets,
                    target_lengths
                )
            )

    elapsed = time.time() - start_time

    evaluation_cer = cer(
        references,
        predictions
    )

    evaluation_wer = wer(
        references,
        predictions
    )

    exact_accuracy = (
        sum(
            r == p
            for r, p in zip(
                references,
                predictions
            )
        ) / len(references)
        if references
        else 0.0
    )

    return {
        "cer": float(evaluation_cer),
        "wer": float(evaluation_wer),
        "exact_accuracy": float(exact_accuracy),
        "references": references,
        "predictions": predictions,
        "evaluation_time_seconds": float(elapsed),
        "samples_per_second": float(
            len(references) / elapsed
            if elapsed > 0
            else 0
        )
    }


def save_predictions(
    references,
    predictions,
    output_path
):

    pd.DataFrame({
        "sample_index": range(len(references)),
        "ground_truth": references,
        "ocr_prediction": predictions
    }).to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )


def save_metrics(
    metrics,
    output_path
):

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metrics,
            f,
            ensure_ascii=False,
            indent=2
        )


with open(
    f"{DATA_ROOT}/dataset_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "dataset": "upanzi/fidel-dataset",
            "sample_type": "handwritten",
            "seed": SEED,
            "image_height": IMG_HEIGHT,
            "image_width": IMG_WIDTH,
            "batch_size": BATCH_SIZE,
            "vocabulary_size": len(characters),
            "training_samples": len(train_df),
            "validation_samples": len(val_df),
            "training_writers": len(train_writers),
            "validation_writers": len(val_writers),
            "writer_disjoint": True,
            "train_split": train_split_path,
            "validation_split": val_split_path,
            "vocabulary": vocab_path
        },
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    f"{STATE_ROOT}/runtime_state.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "seed": SEED,
            "device": str(DEVICE),
            "gpu": (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
            "torch_version": torch.__version__,
            "research_root": RESEARCH_ROOT
        },
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    f"Ready | {DEVICE} | "
    f"{len(train_df)} train | "
    f"{len(val_df)} val | "
    f"{len(characters)} chars"
)

In [ ]:
#@title CRNN baseline

STAGE_ROOT = f"{OCR_RESEARCH_ROOT}/crnn/extended"

os.makedirs(STAGE_ROOT, exist_ok=True)

MAX_EPOCHS = 20
LEARNING_RATE = 1e-3
PATIENCE = 4
NUM_SAMPLE_PREDICTIONS = 3

CONFIG_PATH = f"{STAGE_ROOT}/config.json"
HISTORY_PATH = f"{STAGE_ROOT}/epoch_history.csv"
BEST_PATH = f"{STAGE_ROOT}/best_checkpoint.pt"
LAST_PATH = f"{STAGE_ROOT}/last_checkpoint.pt"
METRICS_PATH = f"{STAGE_ROOT}/metrics.json"
PREDICTIONS_PATH = f"{STAGE_ROOT}/predictions.csv"
SAMPLES_PATH = f"{STAGE_ROOT}/sample_predictions.txt"
README_PATH = f"{STAGE_ROOT}/README.md"


class CRNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(
                kernel_size=(2, 1),
                stride=(2, 1)
            ),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(
                kernel_size=(2, 1),
                stride=(2, 1)
            )
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.2
        )

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, x):

        x = self.cnn(x)

        x = F.adaptive_avg_pool2d(
            x,
            (1, x.size(3))
        )

        x = x.squeeze(2)

        x = x.permute(
            2,
            0,
            1
        )

        x, _ = self.rnn(x)

        x = self.classifier(x)

        return x


model = CRNN(
    num_classes=len(characters) + 1
).to(DEVICE)

criterion = nn.CTCLoss(
    blank=blank_idx,
    zero_infinity=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)


start_epoch = 1
best_cer = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []


if os.path.exists(LAST_PATH):

    checkpoint = torch.load(
        LAST_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    start_epoch = checkpoint["epoch"] + 1

    best_cer = checkpoint.get(
        "best_cer",
        float("inf")
    )

    best_epoch = checkpoint.get(
        "best_epoch",
        0
    )

    epochs_without_improvement = checkpoint.get(
        "epochs_without_improvement",
        0
    )

    if os.path.exists(HISTORY_PATH):

        history = pd.read_csv(
            HISTORY_PATH
        ).to_dict("records")

    print(f"Resuming | epoch {start_epoch}")

else:

    print("Starting CRNN")


config = {
    "model": "CRNN",
    "image_height": IMG_HEIGHT,
    "image_width": IMG_WIDTH,
    "vocabulary_size": len(characters),
    "num_classes": len(characters) + 1,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "patience": PATIENCE,
    "sample_predictions_per_epoch": NUM_SAMPLE_PREDICTIONS,
    "optimizer": "AdamW",
    "weight_decay": 1e-4,
    "loss": "CTCLoss",
    "seed": SEED,
    "train_samples": len(train_df),
    "validation_samples": len(val_df)
}


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )


def save_checkpoint(
    path,
    epoch,
    best_cer,
    best_epoch,
    epochs_without_improvement
):

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_cer": best_cer,
            "best_epoch": best_epoch,
            "epochs_without_improvement": epochs_without_improvement,
            "char_to_idx": char_to_idx,
            "idx_to_char": idx_to_char,
            "blank_idx": blank_idx,
            "config": config
        },
        path
    )


def save_stage_readme():

    with open(
        README_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "# CRNN Extended Training\n\n"
            "Initial CRNN OCR training stage for handwritten "
            "Amharic text recognition.\n\n"
            "## Configuration\n\n"
            f"- Training samples: {len(train_df)}\n"
            f"- Validation samples: {len(val_df)}\n"
            f"- Image size: {IMG_WIDTH}x{IMG_HEIGHT}\n"
            f"- Batch size: {BATCH_SIZE}\n"
            f"- Maximum epochs: {MAX_EPOCHS}\n"
            f"- Optimizer: AdamW\n"
            f"- Learning rate: {LEARNING_RATE}\n"
            f"- Validation split: writer-disjoint\n"
            f"- Sample predictions per epoch: {NUM_SAMPLE_PREDICTIONS}\n"
        )


save_stage_readme()


for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1
):

    model.train()

    running_loss = 0.0
    batch_count = 0

    for images, targets, target_lengths in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(images)

        input_lengths = torch.full(
            size=(images.size(0),),
            fill_value=outputs.size(0),
            dtype=torch.long,
            device=DEVICE
        )

        loss = criterion(
            outputs.log_softmax(2),
            targets,
            input_lengths,
            target_lengths.to(DEVICE)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        running_loss += loss.item()
        batch_count += 1

    train_loss = running_loss / batch_count

    evaluation = evaluate_model(
        model,
        val_loader
    )

    val_cer = evaluation["cer"]
    val_wer = evaluation["wer"]
    exact_accuracy = evaluation["exact_accuracy"]

    scheduler.step(val_cer)

    improved = val_cer < best_cer

    if improved:

        best_cer = val_cer
        best_epoch = epoch
        epochs_without_improvement = 0

        save_checkpoint(
            BEST_PATH,
            epoch,
            best_cer,
            best_epoch,
            epochs_without_improvement
        )

        save_predictions(
            evaluation["references"],
            evaluation["predictions"],
            PREDICTIONS_PATH
        )

        sample_lines = []

        for reference, prediction in zip(
            evaluation["references"][:20],
            evaluation["predictions"][:20]
        ):

            sample_lines.append(
                f"GT: {reference}\n"
                f"PR: {prediction}\n"
            )

        with open(
            SAMPLES_PATH,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                "\n".join(sample_lines)
            )

    else:

        epochs_without_improvement += 1

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_cer": val_cer,
        "val_wer": val_wer,
        "exact_accuracy": exact_accuracy,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "best_cer": best_cer,
        "best_epoch": best_epoch
    }

    history.append(epoch_record)

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )

    save_checkpoint(
        LAST_PATH,
        epoch,
        best_cer,
        best_epoch,
        epochs_without_improvement
    )

    save_metrics(
        {
            "current_epoch": epoch,
            "best_epoch": best_epoch,
            "best_cer": best_cer,
            "current_cer": val_cer,
            "current_wer": val_wer,
            "current_exact_accuracy": exact_accuracy
        },
        METRICS_PATH
    )

    print(
        f"\nEpoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"CER {val_cer:.4f} | "
        f"WER {val_wer:.4f}"
        + (" | best" if improved else "")
    )

    for reference, prediction in zip(
        evaluation["references"][:NUM_SAMPLE_PREDICTIONS],
        evaluation["predictions"][:NUM_SAMPLE_PREDICTIONS]
    ):

        print(f"GT: {reference}")
        print(f"PR: {prediction}")

    if epochs_without_improvement >= PATIENCE:

        print(
            f"Early stop | best epoch {best_epoch}"
        )

        break


print(
    f"\nDone | best epoch {best_epoch} | "
    f"CER {best_cer:.4f}"
)

Resuming | epoch 2

Epoch 02 | loss 1.3214 | CER 0.4350 | WER 1.0110
GT: መጽሐፍ  ቅዱስ  ጻድቃን  ሆይ ፤ በእግዚአብሔር  ደስ ይበላችሁ  ፤ ሐሤትም አድርጉ  ፤ ቅን ልብ ያላች

PR: ፤ መደሐፍ  ቅድሽ  ዳድቃን  ሆሮ  ፤ በእግዚአክሎር  ደስ ይበዳችሁ  ፤ ልሜትም  እድርች  ፤ ቅን መ ያላች

GT: ሁም  ሁሉ እልል  በሉ በማለት  ስለሚያሳስበን  ይህን ዓይነቱን  የልብ  ዝንባሌ መያዝ  እውነተኛ  

PR: ሁም  ሁቡ  እልል  በቡ በማበች  ስለሚያ ስበን  ይህን  ዓይነቹን  የልክ  ዝንባቤ  መያዝ  እነተኛ

GT: ደስታ  ያስገኛል።  የአንጃው  ተወካይና  ከፍተኛ  አራማጅ  ከቤሩት  ተጉዘው በጋዜጣዊ  መግለጫው  

PR: የልጀታ  ያስገኛል።  የእንኛው  ተወካይና  ከፍ ኛ  እልማጀ  ከቤነት  ተገጉዘው  በገዜጣዊ  መግበጫው


Epoch 03 | loss 1.0781 | CER 0.2255 | WER 0.5833 | best
GT: መጽሐፍ  ቅዱስ  ጻድቃን  ሆይ ፤ በእግዚአብሔር  ደስ ይበላችሁ  ፤ ሐሤትም አድርጉ  ፤ ቅን ልብ ያላች

PR: መጽሐፍ  ቅዱስ  ዳድቃን  ሆይ  ፤ በእግዚአብሔር  ደስይብላችሁ  ፤ ሐሤትም  አድርት  ፤ ቅንልብ ያዛች

GT: ሁም  ሁሉ እልል  በሉ በማለት  ስለሚያሳስበን  ይህን ዓይነቱን  የልብ  ዝንባሌ መያዝ  እውነተኛ  

PR: ሁም  ሁሉ እልል  በሉ በማለት  ስለሚያሰሰበን  ይህን  ዓይነቱን  የልብ  ዝንባሎ  መያዝ  እውነተኛ

GT: ደስታ  ያስገኛል።  የአንጃው  ተወካይና  ከፍተኛ  አራማጅ  ከቤሩት  ተጉዘው በጋዜጣዊ  መግለጫው  

PR: ደስጅታ  ያሰገኛል።  የአንጃው  ተወካይና  ከፍተኛ  አልማጅ  ከቤሩት  ተዝዛው  በገዜጣዊ  መግለጫው


Epoch 04 | loss 0.986